# 01 - Source to Bronze with PySpark

Preview and profile the source with Spark SQL, then land an immutable Parquet batch in local storage or Cloudflare R2.

In [ ]:
import sys
from dataclasses import asdict
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT)) if str(ROOT) not in sys.path else None

from insightsstream.spark_pipeline import create_spark_session, ingest_source, read_source
from insightsstream.storage import create_store

spark = create_spark_session("InsightsStream-Notebook-Bronze")
store = create_store()
SOURCE_PATH = ROOT / "data" / "source" / "sample_events.csv"

In [ ]:
source = read_source(spark, SOURCE_PATH)
source.show(20, truncate=False)
source.printSchema()
source.createOrReplaceTempView("source_events")

In [ ]:
spark.sql("""
    SELECT
        lower(trim(metric)) AS metric,
        COUNT(*) AS source_rows,
        ROUND(SUM(CAST(value AS DOUBLE)), 2) AS source_value
    FROM source_events
    GROUP BY lower(trim(metric))
    ORDER BY source_rows DESC
""").show(truncate=False)

In [ ]:
run = ingest_source(SOURCE_PATH, spark=spark, store=store)
print(asdict(run))